In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc samplomatic
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Szybki start z Executor

<Accordion>
<AccordionItem title="Wersje pakietów">

Kod na tej stronie został opracowany przy użyciu następujących wymagań.
Zalecamy używanie tych lub nowszych wersji.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
samplomatic~=0.18.0
```
</AccordionItem>
</Accordion>

Podobnie jak prymityw [Sampler](/guides/get-started-with-sampler), Executor pobiera próbki z rejestrów wyjściowych wynikających z wykonania obwodów kwantowych, ale nie posiada żadnych wbudowanych mechanizmów tłumienia ani łagodzenia błędów. Zamiast tego jest częścią [modelu ukierunkowanego wykonania](/guides/directed-execution-model), który dostarcza składniki do rejestrowania intencji projektowych po stronie klienta i przenosi kosztowne generowanie wariantów obwodów na stronę serwera. Executor wykonuje dyrektywy zawarte w adnotacjach obwodów i opcjach, generuje i wiąże wartości parametrów, wykonuje powiązane obwody na sprzęcie i zwraca wyniki wykonania oraz metadane. Nie podejmuje żadnych niejawnych decyzji i daje ci pełną kontrolę i przejrzystość.

> **Note:** Pakiet Qiskit nie ma jeszcze klasy bazowej dla prymitywu Executor.

## Przed rozpoczęciem
Niektóre przykłady kodu na tej stronie używają `samplex`, który jest częścią pakietu Samplomatic. Dlatego przed uruchomieniem tych bloków kodu musisz zainstalować Samplomatic, jak pokazano w poniższym bloku kodu. Aby uzyskać więcej informacji, zobacz [dokumentację Samplomatic](https://qiskit.github.io/samplomatic).

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, Executor
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from samplomatic.transpiler import generate_boxing_pass_manager
from samplomatic import build

# Initialize the service and choose a backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

In [2]:
print(backend)

<IBMBackend('ibm_fez')>


### 2. Create and transpile a circuit

You need at least one circuit to use the Executor primitive.  It can optionally have parameters.

In [3]:
# Generate the circuit
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.h(1)
circuit.cz(0, 1)
circuit.h(1)

# Using `measure_all` automatically creates the necessary
# classical registers.
circuit.measure_all()

The circuit needs to be transformed to only use instructions supported by the QPU (referred to as *instruction set architecture (ISA)* circuits). Use the transpiler to do this.

In [4]:
# Transpile the circuit
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)
isa_circuit = preset_pass_manager.run(circuit)

### 2. Tworzenie i transpilacja obwodu
Do użycia prymitywu Executor potrzebujesz co najmniej jednego obwodu. Opcjonalnie może on zawierać parametry.

In [5]:
# Initialize an empty program
program = QuantumProgram(shots=25)

# Append the circuit to the program
program.append_circuit_item(isa_circuit)

Obwód musi zostać przekształcony tak, aby używał tylko instrukcji obsługiwanych przez QPU (określanych jako obwody *architektury zestawu instrukcji (ISA)*). Użyj Transpilatora, aby to zrobić.

In [6]:
# Generate a boxing pass manager to group gates
# and measurements into boxes and add
# a`Twirl` annotation.
boxes_pm = generate_boxing_pass_manager(
    # Add gate twirling
    enable_gates=True,
    # Add measurement twirling
    enable_measures=True,
)

boxed_circuit = boxes_pm.run(isa_circuit)
boxed_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/get-started-with-executor/extracted-outputs/245a4574-3ce9-4f77-98c8-af32cde8ac01-0.svg" alt="Output of the previous code cell" />

### 3. Inicjalizacja `QuantumProgram`
Zainicjuj `QuantumProgram` ze swoim obciążeniem roboczym. `QuantumProgram` składa się z `QuantumProgramItems`. Zazwyczaj każdy element składa się z obwodu, zestawu wartości parametrów i ewentualnie `samplex` do losowania zawartości obwodu. Aby uzyskać pełne szczegóły, zobacz [Dane wejściowe i wyjściowe Executor](/guides/executor-input-output).

Poniższa komórka inicjuje `QuantumProgram` i określa wykonanie 25 pomiarów. Następnie dołącza transpilowany obwód docelowy.

In [7]:
# Build the template circuit and the samplex
template_circuit, samplex = build(boxed_circuit)

# Append the template circuit and samplex as a `samplex_item`
program.append_samplex_item(
    template_circuit,
    samplex=samplex,
    shape=(num_randomizations := 20,),
)

### 4. Opcjonalnie: Grupowanie bramek i pomiarów w adnotowane bloki
Grupowanie instrukcji w bloki i ich adnotowanie to podstawowy sposób określania intencji. W poniższym przykładzie używamy `generate_boxing_pass_manager` i jego parametrów twirlingowych do grupowania dwukubitowych bramek i pomiarów w bloki oraz stosowania adnotacji twirlingowej.

In [8]:
# Initialize an Executor with the default options
executor = Executor(mode=backend)

# Submit the job
job = executor.run(program)
job

<RuntimeJobV2('d8286580bvlc73d1vmsg', 'executor')>

In [9]:
# Retrieve the result
result = job.result()

### 6. Wywołanie Executor i uzyskanie wyników
Uruchom `QuantumProgram` na backendzie IBM&reg;, używając prymitywu `Executor` z domyślnymi opcjami. Zobacz [Opcje Executor](/guides/executor-options), aby dowiedzieć się o dostępnych opcjach.